In [1]:
import pandas as pd
import numpy as np

In [2]:
customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")

In [3]:
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [4]:
date_col = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in date_col:
    orders[col] = pd.to_datetime(orders[col], errors = 'coerce')

In [5]:
valid_orders =  orders[orders['order_status'] == 'delivered']

In [6]:
print(f"Entries not delivered: {orders.shape[0] - valid_orders.shape[0]}")

Entries not delivered: 2963


In [17]:
# Merged customer and their orders 
orders_customers = valid_orders.merge(customers, on = 'customer_id', how = 'left')

In [18]:
# Merged what items they ordered
orders_items = orders_customers.merge(order_items, on = 'order_id', how = 'left')

In [19]:
# Merged the product(items') description
orders_items_products = orders_items.merge(products, on = 'product_id', how = 'left')

In [20]:
print(valid_orders.shape)
print(orders_items_products.shape)

(96478, 8)
(110197, 26)


In [21]:
orders_items_products.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        2
order_delivered_customer_date       8
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1537
product_name_lenght              1537
product_description_lenght       1537
product_photos_qty               1537
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
dtype: int64

In [22]:
orders_items_products['price'].describe()

count    110197.000000
mean        119.980563
std         182.299446
min           0.850000
25%          39.900000
50%          74.900000
75%         134.170000
max        6735.000000
Name: price, dtype: float64

In [23]:
orders_items_products = orders_items_products[orders_items_products['price'] > 0]

In [26]:
# astype() Better for groupby, csv etc. functions 
orders_items_products['order_month'] = (orders_items_products['order_purchase_timestamp'].dt.to_period('M').astype(str))

In [27]:
orders_items_products['revenue'] = orders_items_products['price']

In [28]:
orders_items_products.duplicated().sum()

0

In [29]:
orders_items_products['revenue'].sum()

13221498.110000001

In [35]:
orders_items_products.shape    #(110197, 28)
orders_items_products['order_id'].nunique()   #96478
orders_items_products['order_item_id'].nunique() #21 

21

In [37]:
# Each order must have just one item there can be multiple quantity of it
orders_items_products[['order_id', 'order_item_id']].duplicated().sum()

0

In [49]:
orders_items_products.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                15
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
customer_unique_id                0
customer_zip_code_prefix          0
customer_city                     0
customer_state                    0
order_item_id                     0
product_id                        0
seller_id                         0
shipping_limit_date               0
price                             0
freight_value                     0
product_category_name             0
product_name_lenght               0
product_description_lenght        0
product_photos_qty                0
product_weight_g                  0
product_length_cm                 0
product_height_cm                 0
product_width_cm                  0
order_month                       0
revenue                     

In [40]:
orders_items_products['product_category_name'] = (orders_items_products['product_category_name'].fillna('Unknwon'))

In [45]:
meta_num_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']

for cols in meta_num_cols:
    orders_items_products[cols] = (orders_items_products[cols].fillna(0))

In [48]:
dim_col = ['product_weight_g','product_length_cm','product_height_cm', 'product_width_cm']

for col in dim_col:
    orders_items_products[col] = (orders_items_products[col].fillna(
        orders_items_products[col].median()
    ))

In [52]:
# Just to be sure 
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders_items_products[col] = pd.to_datetime(
        orders_items_products[col],
        errors='coerce'
    )

In [53]:
orders_items_products.to_csv("cleaned_ecommerce_data.csv", index = False)

### Data Cleaning & Validation Summary

- Converted all date columns to datetime format
- Filtered only delivered orders for revenue analysis
- Merged customers, orders, order_items, and products
- Validated revenue values to avoid double counting
- Created time-based features for trend and cohort analysis

Result:
Final dataset contains only valid, revenue-generating transactions.